<a href="https://colab.research.google.com/github/soleildayana/Apophis-Asteroid-Project/blob/main/analisis/nb4_kepler_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4 — Solver de Kepler: Benchmarking de Métodos Iterativos
## Problema de los Dos Cuerpos — Asteroide (99942) Apophis

**Autor:** Soleil Dayana Niño Murcia — 1033097666  
**Curso:** Mecánica Celeste  
**Fecha:** Mayo 2026

---

> **Objetivo:** Comparar el rendimiento de tres métodos iterativos para resolver la ecuación de Kepler,
> evaluando convergencia, número de iteraciones y tiempo computacional a lo largo de la órbita de Apophis,
> con énfasis en la región del perigeo donde la anomalía verdadera cambia más rápidamente.

## 1. Teoría: La Ecuación de Kepler

### 1.1 Planteamiento

En el **problema de los dos cuerpos** (Kepler), el movimiento de un objeto en una órbita elíptica queda completamente determinado por la **ecuación de Kepler**:

$$\boxed{M = E - e \sin E}$$

donde:
- $M = n(t - t_p)$ es la **anomalía media** ($n = 2\pi/T$ movimiento medio, $t_p$ tiempo de paso por el perigeo)
- $E$ es la **anomalía excéntrica** (variable geométrica ligada a la elipse)
- $e$ es la **excentricidad** de la órbita

### 1.2 ¿Por qué es trascendental?

La ecuación de Kepler es **trascendental**: no puede resolverse de forma cerrada para $E$ dado $M$. Cualquier estrategia de solución debe ser **iterativa**, lo que plantea preguntas de:
- **Convergencia**: ¿el método siempre converge?
- **Velocidad**: ¿cuántas iteraciones necesita?
- **Eficiencia**: ¿cuál es el costo computacional por iteración?

### 1.3 Por qué la convergencia depende de la posición orbital

La dificultad de la ecuación de Kepler varía a lo largo de la órbita. La relación entre $M$ y $E$ es:
$$\frac{dE}{dM} = \frac{1}{1 - e\cos E}$$

Cerca del **perigeo** ($E \approx 0$, $M \approx 0$): $1 - e\cos E \approx 1 - e$ es pequeño para $e$ cercano a 1,
haciendo que la función sea más empinada y la convergencia más lenta.

Para Apophis con $e \approx 0.191$, el perigeo no presenta dificultad severa, pero el efecto es observable
y educativamente relevante.

### 1.4 Los tres métodos a comparar

| Método | Orden de convergencia | Característica |
|--------|----------------------|----------------|
| **Newton-Raphson** | Cuadrático | Usa derivada primera; robusto y rápido |
| **Punto Fijo** | Lineal | Simple pero convergencia lenta para $e$ grande |
| **Laguerre-Conway** | Cúbico* | Muy robusto; diseñado específicamente para Kepler |

*Para problemas de Kepler, Laguerre-Conway tiene convergencia de orden 5 en la práctica.

In [ ]:
%pip install -Uq pymcel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import time
from datetime import datetime

import pymcel as pc
from pymcel import constantes as const

print('Librerías cargadas correctamente.')

## 2. Clase `KeplerSolver`

Implementamos una clase que encapsula los tres métodos iterativos.
Cada método retorna `(E, residual_history, cpu_time_us)` donde:
- `E`: solución de la anomalía excéntrica
- `residual_history`: lista de residuos $|M - E_n + e\sin E_n|$ en cada iteración
- `cpu_time_us`: tiempo de CPU en microsegundos

In [ ]:
class KeplerSolver:
    """
    Benchmarking de tres métodos iterativos para la ecuación de Kepler:
        M = E - e*sin(E)
    """

    @staticmethod
    def _residual(E, M, e):
        """Residuo: |M - E + e*sin(E)|"""
        return abs(M - E + e * np.sin(E))

    def newton_raphson(self, M, e, tol=1e-10, max_iter=50):
        """
        Método de Newton-Raphson aplicado a f(E) = E - e*sin(E) - M = 0.
        Iteración: E_{n+1} = E_n - f(E_n)/f'(E_n)
                           = E_n - (E_n - e*sin(E_n) - M) / (1 - e*cos(E_n))
        Convergencia cuadrática.
        """
        t0 = time.perf_counter()

        # Estimación inicial estándar (Danby 1988)
        E = M + e * np.sin(M) / (1 - np.sin(M + e) + np.sin(M))
        if not np.isfinite(E):
            E = M

        history = []
        converged = False
        fp_eps = 1e-12
        max_step = np.pi / 2
        for _ in range(max_iter):
            f  = E - e * np.sin(E) - M
            fp = 1 - e * np.cos(E)

            if abs(fp) < fp_eps:
                # Evita divisiones por valores casi nulos y resembra
                E = 0.5 * (E + M)
            else:
                dE = -f / fp
                dE = np.clip(dE, -max_step, max_step)
                E_next = E + dE
                if np.isfinite(E_next):
                    E = E_next
                else:
                    E = 0.5 * (E + M)

            res = self._residual(E, M, e)
            history.append(res)
            if res < tol:
                converged = True
                break

        cpu_us = (time.perf_counter() - t0) * 1e6
        if not converged:
            raise RuntimeError(
                f"Newton-Raphson no convergió en {max_iter} iteraciones para M={M}, e={e}."
            )
        return E, history, cpu_us

    def punto_fijo(self, M, e, tol=1e-10, max_iter=200):
        """
        Método de Punto Fijo: E_{n+1} = M + e*sin(E_n).
        Convergencia lineal (tasa = e < 1).
        """
        t0 = time.perf_counter()

        E = M  # estimación inicial simple
        history = []
        for _ in range(max_iter):
            E_new = M + e * np.sin(E)
            res = self._residual(E_new, M, e)
            history.append(res)
            E = E_new
            if res < tol:
                break

        cpu_us = (time.perf_counter() - t0) * 1e6
        return E, history, cpu_us

    def laguerre_conway(self, M, e, tol=1e-10, max_iter=50, n=5):
        """
        Método de Laguerre-Conway para la ecuación de Kepler.
        Diseñado específicamente para convergencia global y robusta.

        Para f(E) = E - e*sin(E) - M, la iteración de Laguerre de orden n es:
            E_{k+1} = E_k - n*f / (f' ± sqrt(|(n-1)^2*(f')^2 - n*(n-1)*f*f''|))
        donde f'' = e*sin(E) y el signo ± se elige para maximizar el denominador.
        """
        t0 = time.perf_counter()

        E = M + e * np.sin(M) / (1 - np.sin(M + e) + np.sin(M))
        if not np.isfinite(E):
            E = M

        history = []
        for _ in range(max_iter):
            f   = E - e * np.sin(E) - M
            fp  = 1 - e * np.cos(E)
            fpp = e * np.sin(E)

            discriminant = (n - 1)**2 * fp**2 - n * (n - 1) * f * fpp
            sqrt_disc = np.sqrt(abs(discriminant))

            # Elegir signo que maximiza el denominador (mayor estabilidad)
            denom = fp + np.sign(fp) * sqrt_disc
            if abs(denom) < 1e-15:
                denom = fp - np.sign(fp) * sqrt_disc

            dE = -n * f / denom
            E += dE
            res = self._residual(E, M, e)
            history.append(res)
            if res < tol:
                break

        cpu_us = (time.perf_counter() - t0) * 1e6
        return E, history, cpu_us


solver = KeplerSolver()
print('Clase KeplerSolver creada exitosamente.')

# Test rápido
M_test, e_test = np.pi / 3, 0.5
E_nr,  h_nr,  t_nr  = solver.newton_raphson(M_test, e_test)
E_pf,  h_pf,  t_pf  = solver.punto_fijo(M_test, e_test)
E_lag, h_lag, t_lag = solver.laguerre_conway(M_test, e_test)
print(f'Test M={M_test:.4f}, e={e_test}:')
print(f'  Newton-Raphson : E={E_nr:.10f} ({len(h_nr):2d} iter, {t_nr:.2f} μs)')
print(f'  Punto Fijo     : E={E_pf:.10f} ({len(h_pf):2d} iter, {t_pf:.2f} μs)')
print(f'  Laguerre-Conway: E={E_lag:.10f} ({len(h_lag):2d} iter, {t_lag:.2f} μs)')

## 3. Elementos orbitales de Apophis

Obtenemos los elementos orbitales de Apophis en dos épocas:
- **Pre-encuentro:** 2028-01-01 (más de un año antes del acercamiento)
- **Post-encuentro:** 2029-06-01 (≈ 7 semanas después del acercamiento del 13 de abril de 2029)

In [ ]:
# ── Constantes físicas ──────────────────────────────────────────────────────
AU_km    = 149_597_870.7
AU_m     = AU_km * 1e3
M_sun_kg = 1.989e30
G_SI     = 6.674e-11
UT_s     = np.sqrt(AU_m**3 / (G_SI * M_sun_kg))
UT_days  = UT_s / 86_400.0
vel_unit = AU_km / UT_s   # AU/UT en km/s

# μ solar en unidades canónicas (G=1, M_sol=1)
mu_canon = 1.0

print(f'UT = {UT_days:.4f} días')
print(f'1 AU/UT = {vel_unit:.4f} km/s')

# ── Consulta de efemérides de Apophis ─────────────────────────────────────
EPOCAS = {
    'pre':  '2028-01-01',
    'post': '2029-06-01',
}

elementos_apophis = {}
for etiqueta, fecha in EPOCAS.items():
    print(f'Consultando Apophis ({etiqueta}) en {fecha}...', end=' ')
    tabla, jd, estado = pc.consulta_horizons(
        id       = '99942',
        location = '@0',
        epochs   = fecha,
    )
    # estado = [x, y, z, vx, vy, vz] en m y m/s (SPICE) o AU y AU/día (astroquery)
    r_vec = estado[:3]
    v_vec = estado[3:]

    # Si las velocidades están en m/s, convertir; si en km/s, convertir
    # pc.consulta_horizons en modo 'vectors' retorna AU y AU/día via astroquery
    # Convertir v de AU/día → AU/UT
    v_vec_canon = v_vec * UT_days

    # Calcular elementos orbitales
    estado_canon = np.concatenate([r_vec, v_vec_canon])
    elems = pc.estado_a_elementos(mu_canon, estado_canon)
    p, e, inc, Omega, omega, f = elems
    a = p / (1 - e**2)
    h = np.linalg.norm(np.cross(r_vec, v_vec_canon))

    elementos_apophis[etiqueta] = {
        'fecha': fecha, 'a': a, 'e': e, 'i': np.degrees(inc),
        'Omega': np.degrees(Omega), 'omega': np.degrees(omega),
        'f': np.degrees(f), 'h': h,
    }
    print(f'OK | a={a:.4f} AU, e={e:.4f}, i={np.degrees(inc):.2f}°')

print('\nElementos orbitales de Apophis:')
for etq, d in elementos_apophis.items():
    print(f"  [{etq.upper():4s}] {d['fecha']} | a={d['a']:.5f} AU, e={d['e']:.6f}, "
          f"i={d['i']:.4f}°, ω={d['omega']:.3f}°")

## 4. Benchmark: barrido de anomalía media con énfasis en el perigeo

Evaluamos los tres métodos para $M \in [0, 2\pi]$.
Para simular el mayor tiempo que Apophis pasa lejos del perigeo (ley de las áreas),
incluimos un barrido **uniforme** en $M$ (que ya refleja la distribución temporal real).

In [ ]:
N_PUNTOS = 360  # puntos del barrido
M_arr = np.linspace(0.001, 2 * np.pi - 0.001, N_PUNTOS)  # evitar exactamente 0 y 2π

resultados = {}

for etiqueta in ['pre', 'post']:
    e_val = elementos_apophis[etiqueta]['e']
    print(f'Benchmarking [{etiqueta.upper()}] e = {e_val:.6f} ...')

    data = {
        'M':         M_arr,
        'E_NR':      np.zeros(N_PUNTOS),
        'n_iter_NR': np.zeros(N_PUNTOS, dtype=int),
        'res_NR':    np.zeros(N_PUNTOS),
        'cpu_NR':    np.zeros(N_PUNTOS),
        'E_PF':      np.zeros(N_PUNTOS),
        'n_iter_PF': np.zeros(N_PUNTOS, dtype=int),
        'res_PF':    np.zeros(N_PUNTOS),
        'cpu_PF':    np.zeros(N_PUNTOS),
        'E_LAG':     np.zeros(N_PUNTOS),
        'n_iter_LAG': np.zeros(N_PUNTOS, dtype=int),
        'res_LAG':   np.zeros(N_PUNTOS),
        'cpu_LAG':   np.zeros(N_PUNTOS),
    }

    for k, M_val in enumerate(M_arr):
        E, h, t = solver.newton_raphson(M_val, e_val)
        data['E_NR'][k]      = E
        data['n_iter_NR'][k] = len(h)
        data['res_NR'][k]    = h[-1] if h else np.nan
        data['cpu_NR'][k]    = t

        E, h, t = solver.punto_fijo(M_val, e_val)
        data['E_PF'][k]      = E
        data['n_iter_PF'][k] = len(h)
        data['res_PF'][k]    = h[-1] if h else np.nan
        data['cpu_PF'][k]    = t

        E, h, t = solver.laguerre_conway(M_val, e_val)
        data['E_LAG'][k]      = E
        data['n_iter_LAG'][k] = len(h)
        data['res_LAG'][k]    = h[-1] if h else np.nan
        data['cpu_LAG'][k]    = t

    resultados[etiqueta] = data

print('\nBenchmark completado.')

## 5. Tabla resumen

In [ ]:
print('=' * 70)
print(f'{"Época":<8} {"Método":<18} {"Iter. media":>12} {"Iter. máx":>10} {"CPU media (μs)":>16}')
print('=' * 70)

metodos = [
    ('Newton-Raphson', 'NR'),
    ('Punto Fijo',     'PF'),
    ('Laguerre-Conway', 'LAG'),
]

tabla_rows = []
for etq in ['pre', 'post']:
    d = resultados[etq]
    for nombre, clave in metodos:
        n_mean = d[f'n_iter_{clave}'].mean()
        n_max  = d[f'n_iter_{clave}'].max()
        t_mean = d[f'cpu_{clave}'].mean()
        print(f'{etq.upper():<8} {nombre:<18} {n_mean:>12.2f} {n_max:>10d} {t_mean:>16.3f}')
        tabla_rows.append({'Época': etq, 'Método': nombre,
                           'Iter. media': n_mean, 'Iter. máx': n_max,
                           'CPU media (μs)': t_mean})
    print('-' * 70)

df_tabla = pd.DataFrame(tabla_rows)
df_tabla

## 6. Gráficas

### 6.1 Gráfica 1: Número de iteraciones vs. Anomalía Media $M$

Comparamos el número de iteraciones requeridas por cada método a lo largo de la órbita,
destacando la zona del **perigeo** ($M \approx 0$ y $M \approx 2\pi$) donde la ecuación de Kepler es más difícil.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

colores = {'NR': '#1f77b4', 'PF': '#d62728', 'LAG': '#2ca02c'}
labels  = {'NR': 'Newton-Raphson', 'PF': 'Punto Fijo', 'LAG': 'Laguerre-Conway'}

for ax, etq in zip(axes, ['pre', 'post']):
    d = resultados[etq]
    M_deg = np.degrees(d['M'])

    for clave in ['NR', 'PF', 'LAG']:
        ax.plot(M_deg, d[f'n_iter_{clave}'],
                color=colores[clave], label=labels[clave],
                linewidth=1.5, alpha=0.85)

    # Zona del perigeo sombreada
    ax.axvspan(0,  30, color='gold', alpha=0.25, label='Zona perigeo')
    ax.axvspan(330, 360, color='gold', alpha=0.25)
    ax.axvline(180, color='gray', linestyle='--', linewidth=1, label='Apogeo (M=180°)')

    e_val = elementos_apophis[etq]['e']
    ax.set_title(f'Apophis {etq.upper()} — e = {e_val:.4f}', fontsize=12)
    ax.set_xlabel('Anomalía Media $M$ (°)', fontsize=11)
    ax.set_ylabel('Número de iteraciones', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 360)

fig.suptitle('Iteraciones por método vs. Anomalía Media (Apophis)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('kepler_iteraciones_vs_M.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: kepler_iteraciones_vs_M.png')

### 6.2 Gráfica 2: Error residual vs. Tiempo computacional (escala log-log)

Cada punto representa una evaluación del método para un $M$ particular.
El gráfico revela la **eficiencia** de cada método: queremos puntos en la esquina inferior izquierda
(residuo pequeño + tiempo corto).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, etq in zip(axes, ['pre', 'post']):
    d = resultados[etq]

    for clave in ['NR', 'PF', 'LAG']:
        res = d[f'res_{clave}']
        cpu = d[f'cpu_{clave}']
        # Evitar log de cero
        mask = (res > 0) & (cpu > 0)
        ax.scatter(cpu[mask], res[mask],
                   color=colores[clave], label=labels[clave],
                   s=12, alpha=0.6, edgecolors='none')

    ax.set_xscale('log')
    ax.set_yscale('log')
    e_val = elementos_apophis[etq]['e']
    ax.set_title(f'Apophis {etq.upper()} — e = {e_val:.4f}', fontsize=12)
    ax.set_xlabel('Tiempo de CPU (μs)', fontsize=11)
    ax.set_ylabel('Error residual $|M - E + e\\sin E|$', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, which='both')
    ax.axhline(1e-10, color='black', linestyle=':', linewidth=1, label='Tolerancia $10^{-10}$')

fig.suptitle('Error residual vs. Tiempo de CPU (log-log) — Benchmark Kepler', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('kepler_residual_vs_cpu.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: kepler_residual_vs_cpu.png')

### 6.3 Convergencia iteración a iteración

Visualizamos cómo decae el residuo en cada iteración para un valor de $M$ cerca del **perigeo** ($M = 0.1$ rad)
comparado con el **apogeo** ($M = \pi$ rad).

In [ ]:
etq = 'pre'
e_val = elementos_apophis[etq]['e']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
titulos = ['Perigeo ($M = 0.1$ rad)', 'Apogeo ($M = \\pi$ rad)']
M_vals  = [0.1, np.pi]

for ax, M_val, titulo in zip(axes, M_vals, titulos):
    E_nr,  h_nr,  _ = solver.newton_raphson(M_val, e_val)
    E_pf,  h_pf,  _ = solver.punto_fijo(M_val, e_val)
    E_lag, h_lag, _ = solver.laguerre_conway(M_val, e_val)

    for historia, clave in [(h_nr, 'NR'), (h_pf, 'PF'), (h_lag, 'LAG')]:
        iters = np.arange(1, len(historia) + 1)
        # Evitar valores cero para log
        hist_safe = np.where(np.array(historia) > 0, historia, 1e-16)
        ax.semilogy(iters, hist_safe,
                    color=colores[clave], label=labels[clave],
                    marker='o', markersize=4, linewidth=1.8)

    ax.axhline(1e-10, color='black', linestyle=':', linewidth=1.2, label='Tol $10^{-10}$')
    ax.set_title(titulo + f'  (e={e_val:.4f})', fontsize=11)
    ax.set_xlabel('Iteración', fontsize=11)
    ax.set_ylabel('Residuo $|f(E_n)|$', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xlim(left=0)

fig.suptitle('Convergencia iteración a iteración — Apophis PRE-encuentro', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('kepler_convergencia_iteraciones.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: kepler_convergencia_iteraciones.png')

## 7. Conclusiones

### ¿Cuál método converge más rápido cerca del perigeo de Apophis?

In [ ]:
print('=' * 65)
print('CONCLUSIONES DEL BENCHMARK — Ecuación de Kepler para Apophis')
print('=' * 65)

for etq in ['pre', 'post']:
    d    = resultados[etq]
    e_v  = elementos_apophis[etq]['e']
    fecha = elementos_apophis[etq]['fecha']
    print(f'\n[{etq.upper()}] época {fecha}, e = {e_v:.6f}')

    # Zona perigeo: M < 30° o M > 330°
    mask_peri = (d['M'] < np.radians(30)) | (d['M'] > np.radians(330))

    for nombre, clave in metodos:
        n_total = d[f'n_iter_{clave}'].mean()
        n_peri  = d[f'n_iter_{clave}'][mask_peri].mean()
        print(f'  {nombre:<18}: media global = {n_total:.1f} iter | '
              f'media zona perigeo = {n_peri:.1f} iter')

print('\n' + '─' * 65)
print('Resumen:')
print('  • Newton-Raphson: convergencia cuadrática, 3-5 iteraciones típicas.')
print('  • Punto Fijo: convergencia lineal, más lento cerca del perigeo.')
print('  • Laguerre-Conway: convergencia de orden superior, más robusto.')
print(f'  • Con e ≈ {elementos_apophis["pre"]["e"]:.3f}, Apophis no es un caso extremo,')
print('    pero las diferencias entre métodos son observables y educativamente valiosas.')
print('  • Post-encuentro: el cambio en e (perturbación gravitatoria de la Tierra)')
print('    modifica levemente el comportamiento de convergencia.')
print('─' * 65)